# Section 2: Input & Behavior Patterns
## AI-Native Software Architecture — O'Reilly Course

The key idea from the deck:

> **Input is the primary control surface for LLM behavior.**

We will add:
1. Structured prompting
2. Output contracts with validation
3. N-shot examples
4. Prompt versioning and experimentation

In [17]:
import support_utils
from support_utils import (
    call_llm, customer_issues, primary_issue,
    parse_json_response, validate_support_schema, nshot_support_prompt, structured_support_prompt
)
import json, time
from typing import Any, Dict, List

In [2]:
# Toggle LLM mode without editing support_utils.py or restarting the kernel.
# Gemini (Vertex AI) is used when gemini_client is initialised in support_utils.
import support_utils
support_utils.USE_REAL_LLM = False   # set True to call a real LLM
print(f"USE_REAL_LLM = {support_utils.USE_REAL_LLM}")

USE_REAL_LLM = True


## Step 2.1: Structured Prompting

A structured prompt defines:
- the model's role
- the task
- the expected output format
- constraints on what the model should and should not do

In [16]:
structured_output = call_llm(structured_support_prompt(primary_issue), temperature=0.2, force_json=True)

print("=== Structured output ===")
print(structured_output)

KeyboardInterrupt: 

## Step 2.2: Parse and Validate the Output Contract

A prompt contract is only useful if the system validates it.

`parse_json_response` and `validate_support_schema` are imported from `support_utils`
— they are shared infrastructure used by every later section.

In [8]:
parsed, parse_error = parse_json_response(structured_output)

if parse_error:
    print(parse_error)
else:
    validation_errors = validate_support_schema(parsed)
    print("Parsed response:")
    print(parsed)
    print("\nValidation errors:")
    print(validation_errors if validation_errors else "None")

Parsed response:
{'category': 'billing', 'urgency': 'medium', 'next_action': "Verify customer's billing history for a double charge.", 'rationale': 'The customer is reporting an incorrect charge and requesting a refund, which is a medium-urgency billing issue that requires verification.'}

Validation errors:
None


## Step 2.3: N-Shot Prompting

Structured prompts define the format.
Examples define what "good" looks like.

In [9]:
nshot_output = call_llm(nshot_support_prompt(primary_issue), temperature=0.2, force_json=True)
print("=== N-shot output ===")
print(nshot_output)

=== N-shot output ===
{
  "category": "billing",
  "urgency": "high",
  "next_action": "verify the duplicate charge and escalate to a human agent for refund processing",
  "rationale": "The user reports being charged twice for their subscription and is requesting a refund."
}


## Step 2.4: Prompt Versioning & Experiments

Prompts are part of application logic.
They should be versioned, tested, and compared like code.

In [18]:
PROMPT_REGISTRY = {
    "v1_naive": lambda issue: f"Help the user: {issue}",
    "v2_structured": structured_support_prompt,
    "v3_nshot": nshot_support_prompt,
}

def run_prompt_experiment(issue: str, versions: Dict[str, Any]) -> List[Dict[str, Any]]:
    rows = []
    for version, prompt_builder in versions.items():
        prompt = prompt_builder(issue)
        start = time.time()
        output = call_llm(prompt, temperature=0.2,
                          force_json=("structured" in version or "nshot" in version))
        latency_ms = round((time.time() - start) * 1000, 2)
        parsed, parse_error = parse_json_response(output)
        validation_errors = validate_support_schema(parsed) if parsed else []
        rows.append({
            "version": version,
            "latency_ms": latency_ms,
            "is_json": parse_error is None,
            "validation_errors": validation_errors,
            "output_preview": output[:160] + ("..." if len(output) > 160 else ""),
        })
    return rows

def print_experiment_results(rows: List[Dict[str, Any]]) -> None:
    for row in rows:
        print(f"\n### {row['version']}")
        print(f"Latency: {row['latency_ms']} ms")
        print(f"JSON output: {row['is_json']}")
        print(f"Validation errors: {row['validation_errors'] or 'None'}")
        print(f"Preview: {row['output_preview']}")

experiment_results = run_prompt_experiment(primary_issue, PROMPT_REGISTRY)
print_experiment_results(experiment_results)

KeyboardInterrupt: 

## Section 2 takeaway

We did not change the model. We changed the input.

That alone improved:
- structure
- safety
- consistency
- downstream usability

**Next:** Section 3 — grounding outputs with retrieval and memory.